# AI Agents for Real-Time IT Incident Response
### LangChain / LangGraph | Multi-Agent Pipeline
**Agents:** Collector → Screener → Analyzer → Fixer → Feedback Loop

## Cell 1 | Install Dependencies

In [0]:
%pip install langgraph langchain databricks-langchain -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## Cell 2 | LLM Client (Databricks-hosted model)

In [0]:
import mlflow.deployments
# These are pre-built models Databricks hosts
client = mlflow.deployments.get_deploy_client("databricks")
for ep in client.list_endpoints():
    print(ep["name"])

databricks-gpt-5-4
databricks-gpt-5-2
databricks-gpt-oss-120b
databricks-gpt-5-3-codex
databricks-gpt-5-2-codex
databricks-gpt-5-1-codex-max
databricks-gpt-5-1-codex-mini
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-gpt-5-1
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b
databricks-meta-llama-3-1-405b-instruct


### Test

In [0]:
from pprint import pprint

response = client.predict(
    endpoint="databricks-meta-llama-3-3-70b-instruct",  # copy/paste from output above
    inputs={"messages": [{"role": "user", "content": "say hi"}]}
)
pprint(response)

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'content': 'Hi! How can I assist you today?',
                          'role': 'assistant'}}],
 'created': 1773677731,
 'id': 'chatcmpl_46d57508-93cf-4c08-b12a-def1c98a3ccd',
 'model': 'meta-llama-3.3-70b-instruct-121024',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 10, 'prompt_tokens': 37, 'total_tokens': 47}}


In [0]:

from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",  # no endpoint setup
    temperature=0,
)

## Cell 3 | Shared State Schema
The shared `IncidentState` `TypedDict`: all 4 agents read from and write back to this single object.

In [0]:
from typing import TypedDict, List, Optional

class IncidentState(TypedDict):
    raw_alert:   str            # raw payload from monitoring tool
    severity:    Optional[str]  # LOW / MEDIUM / HIGH / CRITICAL
    urgency:     Optional[str]  # ROUTINE / URGENT / IMMEDIATE
    root_cause:  Optional[str]  # output of Analyzer
    remediation: Optional[str]  # output of Fixer
    feedback:    Optional[str]  # engineer notes
    messages:    List[str]      # full trace log

## Cell 4 | Agent Nodes
All 4 agents + feedback node. The Screener uses JSON-mode prompting with a `try/except` fallback so it doesn't blow up on malformed LLM output, a common real-world pain point.

**Create alert queue table (run once)**

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS main.it_ops")
spark.sql("""
    CREATE TABLE IF NOT EXISTS main.it_ops.alert_queue (
        raw_alert  STRING,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        processed  BOOLEAN   DEFAULT FALSE
    )
    TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
""")

DataFrame[]

In [0]:
#---------------------------------------------------------
# Collector Agent:  Data Ingestion Agent. Collects real-time alerts from monitoring tools and formats the alerts for downstream agents.
#---------------------------------------------------------
def collector_agent(state: IncidentState) -> IncidentState:
    """Parse and normalise the raw alert."""
    response = llm.invoke(
        f"You are an IT alert ingestion agent. Parse this raw alert and extract: "
        f"source system, timestamp, affected service, error type, and impact scope.\n\n"
        f"Raw alert: {state['raw_alert']}"
    )
    state["raw_alert"] = response.content 
    state["messages"].append(f"[Collector] {response.content}")
    return state


#---------------------------------------------------------
# Screener Agent: Prioritization Agent. Analyzes incoming alerts to assign severity and urgency levels. 
#---------------------------------------------------------
def screener_agent(state: IncidentState) -> IncidentState:
    """Assign severity and urgency to the normalised alert."""
    response = llm.invoke(
        f"You are an IT severity classification agent. "
        f"Reply in JSON: {{\"severity\": \"LOW|MEDIUM|HIGH|CRITICAL\", \"urgency\": \"ROUTINE|URGENT|IMMEDIATE\"}}.\n\n"
        f"Alert: {state['raw_alert']}"
    )
    import json, re
    try:
        parsed = json.loads(re.search(r"\{.*\}", response.content, re.DOTALL).group())
        state["severity"] = parsed.get("severity", "HIGH")
        state["urgency"]  = parsed.get("urgency",  "URGENT")
    except Exception:
        state["severity"], state["urgency"] = "HIGH", "URGENT"  # safe defaults
    state["messages"].append(f"[Screener] severity={state['severity']}, urgency={state['urgency']}")
    return state


#---------------------------------------------------------
# Analyzer Agent: Analysis Agent. Identifies potential causes for the incident based on logs and metadata. 
#---------------------------------------------------------
def analyzer_agent(state: IncidentState) -> IncidentState:
    """Identify the most likely root cause."""
    response = llm.invoke(
        f"You are a root cause analysis agent. Identify the most likely root cause.\n\n"
        f"Alert: {state['raw_alert']}\n"
        f"Severity: {state['severity']} | Urgency: {state['urgency']}"
    )
    state["root_cause"] = response.content
    state["messages"].append(f"[Analyzer] {response.content}")
    return state


#---------------------------------------------------------
# Fixer Agent: Remediation Agent. Suggests actionable steps for resolving the incident. 
#---------------------------------------------------------
def fixer_agent(state: IncidentState) -> IncidentState:
    """Generate step-by-step remediation actions."""
    response = llm.invoke(
        f"You are a remediation recommendation agent. "
        f"Provide numbered, actionable resolution steps.\n\n"
        f"Root cause: {state['root_cause']}\n"
        f"Severity: {state['severity']}"
    )
    state["remediation"] = response.content
    state["messages"].append(f"[Fixer] {response.content}")
    return state

#---------------------------------------------------------
# Feedback Node: Captures feedback from engineers to improve system performance. 
#---------------------------------------------------------
def feedback_node(state: IncidentState) -> IncidentState:
    from pyspark.sql import Row
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    from datetime import datetime

    spark.sql("CREATE SCHEMA IF NOT EXISTS main.it_ops")  # ensure schema exists

    schema = StructType([
        StructField("alert",             StringType(),   True),
        StructField("severity",          StringType(),   True),
        StructField("urgency",           StringType(),   True),
        StructField("root_cause",        StringType(),   True),
        StructField("remediation",       StringType(),   True),
        StructField("engineer_feedback", StringType(),   True),
        StructField("created_at",        TimestampType(), True),
    ])

    row = Row(
        alert=state["raw_alert"][:500],
        severity=state["severity"],
        urgency=state["urgency"],
        root_cause=state["root_cause"],
        remediation=state["remediation"],
        engineer_feedback=state.get("feedback"),
        created_at=datetime.utcnow(),
    )

    spark.createDataFrame([row], schema=schema) \
         .write.mode("append").saveAsTable("main.it_ops.incident_log")
    return state

## Cell 5 | Build & Compile the Graph
Graph wired via a loop instead of 5 separate `add_node` calls, keeping it tight.

There are 2 axes of agent autonomy:

- **Instructional** autonomy: How to do the task (the LLM decides)
- **Operational** autonomy: Whether and when to do the task (the graph decides)

For the latter, LangGraph supports **conditional edges**: the graph checks a state value after each node and routes accordingly. The Screener's output can influence the path.

The Screener still doesn't decide the route directly: it writes severity to state, and `route_by_severity `reads it. But the effect is the same: the Screener's judgment now has operational consequences. A LOW severity alert never touches the Analyzer.

In [0]:
from langgraph.graph import StateGraph, END

def route_by_severity(state: IncidentState) -> str:
    """The Screener's output now has operational weight."""
    if state.get("severity") in ("LOW", "MEDIUM"):
        return "fixer"       # skip deep analysis
    return "analyzer"        # HIGH / CRITICAL → full path

graph = StateGraph(IncidentState)

for name, fn in [
    ("collector", collector_agent),
    ("screener",  screener_agent),
    ("analyzer",  analyzer_agent),
    ("fixer",     fixer_agent),
    ("feedback",  feedback_node),
]:
    graph.add_node(name, fn)

graph.set_entry_point("collector")
graph.add_edge("collector", "screener")

# Screener now has operational autonomy: it influences the next step
graph.add_conditional_edges("screener", route_by_severity, {
    "analyzer": "analyzer",
    "fixer":    "fixer",
})

graph.add_edge("analyzer", "fixer")
graph.add_edge("fixer",    "feedback")
graph.add_edge("feedback", END)

app = graph.compile()

## Cell 6 | Run the Pipeline
Single `app.invoke()` call with a sample alert, prints severity, root cause, and remediation.

**Hard-coded altert (test)**

In [0]:
result = app.invoke({
    "raw_alert":   "CPU spike to 98% on prod-db-01 at 14:32 UTC. 500 errors on API gateway.",
    "severity":    None,
    "urgency":     None,
    "root_cause":  None,
    "remediation": None,
    "feedback":    None,
    "messages":    [],
})

print("=" * 60)
print(f"Severity : {result['severity']} | Urgency: {result['urgency']}")
print("=" * 60)
print("Root Cause:\n", result["root_cause"])
print("=" * 60)
print("Remediation Steps:\n", result["remediation"])

Severity : CRITICAL | Urgency: IMMEDIATE
Root Cause:
 Based on the provided information, I'll attempt to identify the most likely root cause of the issue.

Given the CPU spike on the database server (prod-db-01) and the 500 errors on the API gateway, it's likely that the root cause is related to an overload or inefficient query on the database server. Here are a few possible root causes:

1. **Inefficient database query**: A poorly optimized database query could be causing the CPU spike on the database server, leading to a cascade of 500 errors on the API gateway as it waits for the database to respond.
2. **Database deadlock or locking issue**: A deadlock or locking issue on the database server could be preventing queries from completing, causing the CPU to spike and leading to 500 errors on the API gateway.
3. **Insufficient database resources**: The database server might be under-provisioned or experiencing a sudden increase in load, causing the CPU to spike and leading to 500 error

**Simulate a monitoring tool pushing an alert (run anytime)**

In [0]:
from datetime import datetime

new_alert = spark.createDataFrame([
    ("CPU spike to 98% on prod-db-01 at 14:32 UTC. 500 errors on API gateway.", datetime.utcnow(), False),
    ("Disk usage at 95% on prod-storage-02. Write latency > 800ms.",             datetime.utcnow(), False),
], ["raw_alert", "created_at", "processed"])

new_alert.write.mode("append").saveAsTable("main.it_ops.alert_queue")
print("Alerts queued. Run process.")

Alerts queued. Run process.


`process_pending_alerts()`

 Each processed alert is marked processed = TRUE so it won't be picked up again. The replace("'", "''") in the UPDATE handles any single quotes in the alert text.

In [0]:
from pyspark.sql.functions import col

def process_pending_alerts():
    pending = spark.table("main.it_ops.alert_queue") \
                   .filter(col("processed") == False)
    rows = pending.collect()

    if not rows:
        print("No pending alerts.")
        return

    for row in rows:
        print(f"\nProcessing: {row.raw_alert[:60]}...")
        result = app.invoke({
            "raw_alert":   row.raw_alert,
            "severity":    None,
            "urgency":     None,
            "root_cause":  None,
            "remediation": None,
            "feedback":    None,
            "messages":    [],
        })
        print(f"Severity : {result['severity']} | Urgency: {result['urgency']}")
        print(f"Root Cause:\n{result['root_cause']}")
        print(f"Remediation:\n{result['remediation']}")

        # Mark as processed
        spark.sql(f"""
            UPDATE main.it_ops.alert_queue
            SET processed = TRUE
            WHERE raw_alert = '{row.raw_alert.replace("'", "''")}'
        """)



In [0]:
process_pending_alerts()


Processing: CPU spike to 98% on prod-db-01 at 14:32 UTC. 500 errors on A...
Severity : HIGH | Urgency: URGENT
Root Cause:
To identify the most likely root cause of the CPU spike on prod-db-01, which is leading to 500 errors on the API gateway, let's consider the key pieces of information provided:

1. **Source System (prod-db-01)**: The issue is originating from the database server, indicating that the problem is likely related to database operations or resource utilization.
2. **Affected Service (API gateway)**: The API gateway is experiencing 500 errors, which suggests that the service is unable to process requests correctly, possibly due to the database being unresponsive or slow.
3. **Error Type (500 errors)**: These are internal server errors, which can be caused by a variety of issues including database connection problems, queries timing out, or the database server being overwhelmed.
4. **Impact Scope**: The impact is currently limited to the API gateway but has the potential t

## Cell 7 | (Optional) Submit Engineer Feedback
Optional feedback function that writes engineer ratings back to the Delta table for future fine-tuning.

In [0]:
# Add the column once
if "feedback_rating" not in [f.name for f in spark.table("main.it_ops.incident_log").schema.fields]:
    spark.sql("ALTER TABLE main.it_ops.incident_log ADD COLUMN feedback_rating INT")

In [0]:
def submit_feedback(incident_result: dict, rating: int, comment: str):
    from pyspark.sql.functions import col, lit, when

    alert_prefix = incident_result["raw_alert"][:100]
    df = spark.table("main.it_ops.incident_log")

    updated = df.withColumn(
        "engineer_feedback",
        when(
            col("alert").startswith(alert_prefix) & col("engineer_feedback").isNull(),
            lit(comment)
        ).otherwise(col("engineer_feedback"))
    ).withColumn(
        "feedback_rating",
        when(
            col("alert").startswith(alert_prefix) & col("feedback_rating").isNull(),
            lit(rating)
        ).otherwise(col("feedback_rating"))
    )

    updated.write.mode("overwrite").saveAsTable("main.it_ops.incident_log")
    print(f"Feedback logged | rating: {rating}/5")


In [0]:
# Usage
submit_feedback(result, rating=4, comment="Good steps, but missed DB connection pool check.")

Feedback logged | rating: 4/5
